# Week 4 Hands-On Lab — Search Under Real Constraints

**ESP3201 · formative hands-on lab.** Runs on free-tier Google Colab with a **CPU runtime** (no GPU needed).

You will run a panel of classic graph-search planners on grid maps and weigh them not by path length alone, but by **search effort (nodes expanded), memory (peak frontier), runtime, and replanning cost** — the realities that decide which planner ships on a robot.

> **Report only what your run actually produced.** Every number in your worksheet must come from a cell you ran.

**Attribution.** Problem framing follows the grid-search pedagogy of Berkeley CS188 (Pacman search projects) and Russell & Norvig, *AIMA*. All lab code is original to this course.

## Tasks

1. Run the planner panel on the `maze` map and read the instrumentation.
2. Compare the panel across `open`, `maze`, and `terrain` maps.
3. Isolate the **heuristic effect**: admissible A* vs inflated heuristic vs greedy.
4. **Sweep the Weighted-A* weight yourself** and watch search effort collapse.
5. Run the **replanning** scenario.
6. Fill the worksheet.

## Setup

In [ ]:
import sys, os

COURSE_REPO_URL = "https://github.com/bingquan/esp3201-public.git"

try:
    import search_lab
except ModuleNotFoundError:
    here = os.getcwd()
    cand = os.path.abspath(os.path.join(here, os.pardir))
    if os.path.exists(os.path.join(cand, "search_lab.py")):
        sys.path.insert(0, cand)
    elif COURSE_REPO_URL:
        os.system(f"git clone -q {COURSE_REPO_URL} course_repo")
        sys.path.insert(0, "course_repo/mini_assignments/week04_planner_tradeoff_study/starter")
    else:
        raise ModuleNotFoundError("search_lab.py not found. On Colab set COURSE_REPO_URL.")
    import search_lab
from search_lab import (
    load_map, render, run_all, replan_on_obstacle, sweep_weight,
    bfs, dfs, ucs, greedy, astar, weighted_astar, inadmissible,
)
print("search_lab loaded.")

In [ ]:
def show(results, title=None):
    """Print a panel of SearchResult objects as an aligned table."""
    if title:
        print(title)
    cols = ["algorithm", "found", "cost", "path_len", "nodes_expanded", "max_frontier", "runtime_ms"]
    rows = [r.as_row() for r in results]
    widths = {c: max(len(c), *(len(str(row[c])) for row in rows)) for c in cols}
    header = "  ".join(c.ljust(widths[c]) for c in cols)
    print(header); print("-" * len(header))
    for row in rows:
        print("  ".join(str(row[c]).ljust(widths[c]) for c in cols))

## Part A — Read the instrumentation on one map

`nodes_expanded` is the CPU proxy; `max_frontier` is the memory proxy. Watch how planners that find the **same path** can differ wildly in effort.

In [ ]:
show(run_all("maze"), title="=== maze ===")
print()
print(render(load_map("maze"), astar(load_map("maze")).path))

## Part B — Compare across map structures

- `open`: lots of free space and ties on the optimal f-contour.
- `maze`: dead ends and corridors.
- `terrain`: non-uniform step costs (digits are entry costs).

In [ ]:
for name in ("open", "maze", "terrain"):
    show(run_all(name), title=f"=== {name} ===")
    print()

**Look for:** a planner that finds a *valid but expensive* path (DFS on `terrain`), and a map where an admissible heuristic does NOT reduce search effort (ties on `open`).

## Part C — The heuristic effect

Same map, three heuristic regimes. An **inflated** heuristic expands fewer nodes; greedy ignores path cost entirely.

In [ ]:
name = "terrain"
panel = [
    ucs(load_map(name)),
    astar(load_map(name)),
    astar(load_map(name), inadmissible(3.0), label="A* inflated (w=3)"),
    greedy(load_map(name)),
    weighted_astar(load_map(name), weight=2.0),
]
show(panel, title=f"=== heuristic effect on {name} ===")

## Part D — Sweep the weight yourself (manipulate and discover)

Weighted A* uses `f = g + weight * h`. `weight = 1.0` is plain A*. **Change the weight and watch what happens** to search effort and to solution cost on a large open field.

In [ ]:
print('weight  nodes_expanded  cost')
for w, r in sweep_weight('open_large', weights=(1.0, 1.5, 2.0, 3.0, 5.0)):
    print(f'{w:>5}   {r.nodes_expanded:>13}   {r.cost:>4}')

**Discover it yourself:** what happened to `nodes_expanded` as the weight rose? What happened to `cost`? On this map Weighted A* kept the optimal cost even at weight 5 — *why might that be* (hint: this implementation re-opens nodes when it finds a cheaper route), and what guarantee does Weighted A* actually give about cost? Try the sweep on `terrain` and `maze` too.

## Part E — Replanning when the world changes

A dynamic obstacle appears on the planned path. Compare replanning **from scratch** vs **from the agent's current cell**.

In [ ]:
grid = load_map("open")
initial = astar(load_map("open"))
obstacle = initial.path[len(initial.path) // 2]
print(f"Dropping obstacle at {obstacle}\n")
init, full, partial = replan_on_obstacle(grid, obstacle, planner=astar)
show([init, full, partial], title="=== initial vs full replan vs partial replan ===")

## Worksheet (your deliverable)

### 1. Comparison table

Pick one map and fill from your runs:

| Planner | Found? | Cost | Nodes expanded | Max frontier | Runtime (ms) |
|---------|--------|------|----------------|--------------|--------------|
| BFS | | | | | |
| DFS | | | | | |
| UCS | | | | | |
| Greedy | | | | | |
| A* (admissible) | | | | | |
| A* (inflated) | | | | | |
| Weighted A* | | | | | |

### 2. Weight-sweep finding (from Part D)

| weight | nodes_expanded | cost |
|--------|----------------|------|
| 1.0 | | |
| 2.0 | | |
| 5.0 | | |

- In one sentence: what did inflating the weight buy you, and what did it (or did it not) cost?

### 3. Two findings (each tied to numbers)

- **Heuristic / search effort:** which choice reduced `nodes_expanded`, and did it cost optimality?
- **Operational weakness:** a planner that was *technically correct* but operationally unattractive, with numbers.

### 4. Planner choice under a constraint

- `Stated constraint:` (e.g. "replan within X ms", "frontier under N", "cost within 10% of optimal")
- `Planner you would deploy:`
- `Tradeoff you are accepting:`
- `Evidence (numbers) that justify it:`

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking